# Experiment A — Classical ML Baseline (CIFAKE)
### Real vs. AI-Generated Image Detection using Handcrafted Features + LightGBM/XGBoost

| Field | Value |
|---|---|
| **Experiment ID** | EXP-A-CLASSICAL-v1.0 |
| **Objective** | Establish a reproducible handcrafted-feature baseline for real vs. synthetic image detection |
| **Dataset** | CIFAKE (Kaggle, Version 3) — 120,000 images, 32×32 RGB |
| **Environment** | Kaggle Notebook, T4 GPU (classical models run on CPU; GPU not required but available) |
| **Random Seed** | 42 |
| **Protocol** | Experiment Protocol v1.0 (fixed dataset split, no augmentation, threshold tuned on validation only) |

**Why handcrafted features?** DCT/HOG/LBP/GLCM/Wavelets capture complementary low-level statistics (frequency artifacts, edge/shape distributions, micro-texture, spatial co-occurrence) that generative models often perturb in detectable ways, independent of any learned CNN/ViT representation. This isolates "what classical signal exists" before we add deep backbones in Experiments B/C.

**Why LightGBM?** Fast, strong on tabular/handcrafted feature vectors, native handling of feature importance and monotonic behavior, GPU-optional. XGBoost is run as a secondary comparison on the same splits/features.

**Why no augmentation here?** Augmentations (crop/rotate/jitter) alter the very statistics these descriptors measure, which would confound the baseline. Augmentation is reserved for Experiments B/C (deep learning).

**PAPER LINK** https://arxiv.org/pdf/2601.19262
This notebook is fully self-contained: run top to bottom on Kaggle with the CIFAKE dataset attached as input.


## 0. Environment & Package Versions

In [ ]:
import subprocess, sys

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install(["lightgbm", "xgboost", "shap", "scikit-image", "PyWavelets", "umap-learn"])

import platform, sklearn, lightgbm, xgboost, shap, cv2
print("Python      :", platform.python_version())
print("scikit-learn:", sklearn.__version__)
print("lightgbm    :", lightgbm.__version__)
print("xgboost     :", xgboost.__version__)
print("shap        :", shap.__version__)
print("opencv      :", cv2.__version__)

# --- Suppress cosmetic, non-functional warnings ---
# LightGBM re-checks feature names on every predict() call against the generic
# names it auto-assigns at fit() time when given a raw numpy array; this fires
# on every predict_proba() call (including inside permutation_importance) but
# has zero effect on predictions or metrics.
import warnings
warnings.filterwarnings("ignore", category=UserWarning, message=".*valid feature names.*")
warnings.filterwarnings("ignore", category=UserWarning, module="shap")
warnings.filterwarnings("ignore", category=FutureWarning)


## 1. Configuration Block
Every tunable parameter lives here. Nothing below this cell should hardcode a value that belongs here.

In [ ]:
import os, json, random
import numpy as np

CONFIG = {
    "experiment_id": "EXP-A-CLASSICAL-v1.0",
    "seed": 42,
    "val_split": 0.10,

    # ---- Dataset paths (edit DATA_ROOT if Kaggle mounts the dataset elsewhere) ----
    "data_root": "/kaggle/input/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images",
    "train_dir": "train",
    "test_dir": "test",
    "classes": ["REAL", "FAKE"],   # REAL -> 0, FAKE -> 1
    "img_size": 32,

    # ---- Feature extraction params ----
    "dct_k": 8,                    # top-left k x k low-freq block per channel
    "hist_bins": 16,               # per-channel color histogram bins
    "hog_cell": (8, 8),
    "hog_block": (2, 2),
    "hog_orientations": 9,
    "lbp_P": 8,
    "lbp_R": 1,
    "lbp_bins": 16,
    "glcm_distances": [1],
    "glcm_angles": [0, np.pi/4, np.pi/2, 3*np.pi/4],
    "glcm_levels": 32,
    "wavelet": "db2",

    # ---- Debug / dev toggle ----
    # Set to an int (e.g. 4000) to subsample for a fast dry run; None = full dataset.
    "subsample_per_class_train": None,
    "subsample_per_class_test": None,

    # ---- Model params ----
    "lgbm_params": {
        "n_estimators": 800,
        "learning_rate": 0.05,
        "num_leaves": 63,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 1.0,
        "objective": "binary",
        "random_state": 42,
        "n_jobs": -1,
    },
    "xgb_params": {
        "n_estimators": 800,
        "learning_rate": 0.05,
        "max_depth": 6,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 1.0,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "random_state": 42,
        "n_jobs": -1,
    },

    # ---- Output ----
    "output_dir": "/kaggle/working/experiment_A",
}

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)

set_all_seeds(CONFIG["seed"])

for sub in ["model", "metrics", "figures", "logs", "shap", "errors", "report", "features"]:
    os.makedirs(os.path.join(CONFIG["output_dir"], sub), exist_ok=True)

with open(os.path.join(CONFIG["output_dir"], "logs", "experiment_config.json"), "w") as f:
    json.dump({k: (v if not isinstance(v, np.ndarray) else v.tolist()) for k, v in CONFIG.items()}, f, indent=2, default=str)

print("Config saved. Output root:", CONFIG["output_dir"])


## 2. Dataset Discovery & Integrity Check
Never assume the dataset is correct — verify counts, corrupted files, and resolution before doing anything else.

In [ ]:
import glob
from PIL import Image
import pandas as pd

def list_dataset(root, split_dir, classes):
    rows = []
    for cls in classes:
        folder = os.path.join(root, split_dir, cls)
        files = glob.glob(os.path.join(folder, "*.*"))
        for fp in files:
            rows.append({"path": fp, "label_str": cls})
    return pd.DataFrame(rows)

train_df = list_dataset(CONFIG["data_root"], CONFIG["train_dir"], CONFIG["classes"])
test_df  = list_dataset(CONFIG["data_root"], CONFIG["test_dir"],  CONFIG["classes"])

print("Train images found:", len(train_df))
print("Test images found :", len(test_df))
print(train_df["label_str"].value_counts())
print(test_df["label_str"].value_counts())

def check_corrupted(df, n_check=500, seed=42):
    sample = df.sample(min(n_check, len(df)), random_state=seed)
    bad = []
    for fp in sample["path"]:
        try:
            with Image.open(fp) as im:
                im.verify()
        except Exception as e:
            bad.append((fp, str(e)))
    return bad

bad_train = check_corrupted(train_df)
bad_test = check_corrupted(test_df)
print(f"Corrupted (sampled) - train: {len(bad_train)}, test: {len(bad_test)}")
assert len(bad_train) == 0 and len(bad_test) == 0, "Corrupted images detected — inspect before proceeding."

with Image.open(train_df["path"].iloc[0]) as im:
    print("Sample resolution:", im.size, im.mode)


## 3. Sample Images

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i, cls in enumerate(CONFIG["classes"]):
    sample_paths = train_df[train_df["label_str"] == cls]["path"].sample(6, random_state=CONFIG["seed"]).tolist()
    for j, fp in enumerate(sample_paths):
        img = Image.open(fp)
        axes[i, j].imshow(img)
        axes[i, j].axis("off")
        if j == 0:
            axes[i, j].set_ylabel(cls)
    axes[i, 0].set_title(cls, loc="left")
plt.suptitle("CIFAKE Samples — Row 0: REAL, Row 1: FAKE")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "figures", "sample_images.png"), dpi=150)
plt.show()


## 4. Train / Validation Split (Stratified, 10%)
Test split is the original CIFAKE test folder — untouched, used only at the very end.

In [ ]:
from sklearn.model_selection import train_test_split

train_df["label"] = (train_df["label_str"] == "FAKE").astype(int)
test_df["label"] = (test_df["label_str"] == "FAKE").astype(int)

if CONFIG["subsample_per_class_train"] is not None:
    train_df = train_df.groupby("label_str", group_keys=False).apply(
        lambda x: x.sample(min(len(x), CONFIG["subsample_per_class_train"]), random_state=CONFIG["seed"])
    ).reset_index(drop=True)
if CONFIG["subsample_per_class_test"] is not None:
    test_df = test_df.groupby("label_str", group_keys=False).apply(
        lambda x: x.sample(min(len(x), CONFIG["subsample_per_class_test"]), random_state=CONFIG["seed"])
    ).reset_index(drop=True)

fit_df, val_df = train_test_split(
    train_df, test_size=CONFIG["val_split"], stratify=train_df["label"], random_state=CONFIG["seed"]
)

split_summary = {
    "train_fit": len(fit_df), "train_fit_class_balance": fit_df["label_str"].value_counts().to_dict(),
    "val": len(val_df), "val_class_balance": val_df["label_str"].value_counts().to_dict(),
    "test": len(test_df), "test_class_balance": test_df["label_str"].value_counts().to_dict(),
    "seed": CONFIG["seed"],
}
print(json.dumps(split_summary, indent=2))
with open(os.path.join(CONFIG["output_dir"], "logs", "split_summary.json"), "w") as f:
    json.dump(split_summary, f, indent=2)


## 5. Preprocessing
Classical ML only — **no random augmentation**, deterministic and identical for train/val/test:
- Load image, ensure RGB
- Normalize pixel values to [0, 1] where needed by a descriptor
- Convert to grayscale where a descriptor requires it (HOG, LBP, GLCM, Wavelets)


## 6. Feature Extraction
Each descriptor implemented per the methodology: Raw pixels, Color Histogram, DCT, HOG, LBP, GLCM, Wavelets.

In [ ]:
import cv2
import pywt
from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops

def load_rgb(path, size=None):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if size is not None:
        img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    return img

def feat_raw_stats(img_rgb):
    # Normalized RGB flattened + basic per-channel stats
    norm = img_rgb.astype(np.float32) / 255.0
    flat = norm.flatten()
    stats = np.concatenate([norm.mean(axis=(0,1)), norm.std(axis=(0,1)),
                             norm.min(axis=(0,1)), norm.max(axis=(0,1))])
    return np.concatenate([flat, stats])

def feat_color_hist(img_rgb, bins=16):
    hists = []
    for c in range(3):
        h, _ = np.histogram(img_rgb[:, :, c], bins=bins, range=(0, 255), density=False)
        h = h.astype(np.float32)
        h = h / (h.sum() + 1e-8)  # L1 normalize
        hists.append(h)
    return np.concatenate(hists)

def feat_dct(img_rgb, k=8):
    out = []
    for c in range(3):
        ch = img_rgb[:, :, c].astype(np.float32) / 255.0
        d = cv2.dct(ch)
        block = d[:k, :k].flatten()
        out.append(block)
    return np.concatenate(out)

def feat_hog(gray, cell=(8, 8), block=(2, 2), orientations=9):
    return hog(gray, orientations=orientations, pixels_per_cell=cell,
               cells_per_block=block, block_norm="L2-Hys", feature_vector=True)

def feat_lbp(gray, P=8, R=1, bins=16):
    lbp = local_binary_pattern(gray, P=P, R=R, method="uniform")
    hist, _ = np.histogram(lbp, bins=bins, range=(0, P + 2), density=True)
    return hist.astype(np.float32)

def feat_glcm(gray, distances, angles, levels=32):
    q = (gray.astype(np.float64) / 256.0 * levels).astype(np.uint8)
    q = np.clip(q, 0, levels - 1)
    glcm = graycomatrix(q, distances=distances, angles=angles, levels=levels, symmetric=True, normed=True)
    props = ["contrast", "homogeneity", "energy", "correlation"]
    out = np.concatenate([graycoprops(glcm, p).flatten() for p in props])
    return out

def feat_wavelet(gray, wavelet="db2"):
    coeffs2 = pywt.dwt2(gray.astype(np.float32) / 255.0, wavelet)
    A, (LH, HL, HH) = coeffs2
    def energy(b): return np.mean(b ** 2)
    return np.array([energy(LH), energy(HL), energy(HH), A.mean(), A.std()], dtype=np.float32)

def extract_all_features(path, cfg):
    img_rgb = load_rgb(path, size=cfg["img_size"])
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

    f_raw = feat_raw_stats(img_rgb)
    f_hist = feat_color_hist(img_rgb, bins=cfg["hist_bins"])
    f_dct = feat_dct(img_rgb, k=cfg["dct_k"])
    f_hog = feat_hog(gray, cell=cfg["hog_cell"], block=cfg["hog_block"], orientations=cfg["hog_orientations"])
    f_lbp = feat_lbp(gray, P=cfg["lbp_P"], R=cfg["lbp_R"], bins=cfg["lbp_bins"])
    f_glcm = feat_glcm(gray, distances=cfg["glcm_distances"], angles=cfg["glcm_angles"], levels=cfg["glcm_levels"])
    f_wav = feat_wavelet(gray, wavelet=cfg["wavelet"])

    return {
        "baseline": np.concatenate([f_raw, f_hist, f_dct]),
        "advanced": np.concatenate([f_hog, f_lbp, f_glcm, f_wav]),
        "mixed": np.concatenate([f_raw, f_hist, f_dct, f_hog, f_lbp, f_glcm, f_wav]),
    }

# quick shape check on one sample
_sample_feats = extract_all_features(train_df["path"].iloc[0], CONFIG)
for k, v in _sample_feats.items():
    print(f"{k:10s} dim = {v.shape[0]}")


In [ ]:
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

def _worker(args):
    path, cfg = args
    feats = extract_all_features(path, cfg)
    return feats

def build_feature_matrix(df, cfg, desc=""):
    paths = df["path"].tolist()
    labels = df["label"].values
    n = len(paths)

    dims = {k: v.shape[0] for k, v in _sample_feats.items()}
    out = {k: np.zeros((n, d), dtype=np.float32) for k, d in dims.items()}

    for i, path in enumerate(tqdm(paths, desc=desc)):
        feats = extract_all_features(path, cfg)
        for k in out:
            out[k][i] = feats[k]

    return out, labels

# NOTE: sequential extraction (safe on Kaggle). For the full 100k/10k/20k images this
# takes a while — use CONFIG["subsample_per_class_train"/"test"] for a fast dry run first.
feat_fit, y_fit = build_feature_matrix(fit_df, CONFIG, desc="Extracting FIT features")
feat_val, y_val = build_feature_matrix(val_df, CONFIG, desc="Extracting VAL features")
feat_test, y_test = build_feature_matrix(test_df, CONFIG, desc="Extracting TEST features")

feat_dir = os.path.join(CONFIG["output_dir"], "features")
for split_name, feats, labels in [("fit", feat_fit, y_fit), ("val", feat_val, y_val), ("test", feat_test, y_test)]:
    np.savez_compressed(os.path.join(feat_dir, f"{split_name}.npz"),
                         labels=labels, **{f"X_{k}": v for k, v in feats.items()})
print("Feature extraction complete. Saved to", feat_dir)


## 7. Feature Analysis
Sanity-check the extracted features before training: variance, correlation, PCA separability.

In [ ]:
from sklearn.decomposition import PCA

def feature_analysis(X, y, name, out_dir):
    variances = X.var(axis=0)
    plt.figure(figsize=(8, 3))
    plt.hist(variances, bins=50)
    plt.title(f"Feature variance distribution — {name}")
    plt.xlabel("variance"); plt.ylabel("count")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"feat_variance_{name}.png"), dpi=150)
    plt.close()

    pca = PCA(n_components=2, random_state=CONFIG["seed"])
    Z = pca.fit_transform(X)
    plt.figure(figsize=(6, 5))
    for lab, color, lbl in [(0, "tab:blue", "REAL"), (1, "tab:red", "FAKE")]:
        mask = y == lab
        plt.scatter(Z[mask, 0], Z[mask, 1], s=4, alpha=0.4, c=color, label=lbl)
    plt.legend(); plt.title(f"PCA projection — {name}")
    plt.xlabel("PC1"); plt.ylabel("PC2")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"pca_{name}.png"), dpi=150)
    plt.close()
    return {"n_zero_variance_features": int((variances < 1e-8).sum()), "explained_variance_ratio": pca.explained_variance_ratio_.tolist()}

fig_dir = os.path.join(CONFIG["output_dir"], "figures")
analysis_summary = {}
for name in ["baseline", "advanced", "mixed"]:
    analysis_summary[name] = feature_analysis(feat_fit[name], y_fit, name, fig_dir)

print(json.dumps(analysis_summary, indent=2))
with open(os.path.join(CONFIG["output_dir"], "logs", "feature_analysis.json"), "w") as f:
    json.dump(analysis_summary, f, indent=2)


## 8. Evaluation Utilities
One shared implementation used identically for every model/feature-set combination (fair comparison, no metric drift).

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    matthews_corrcoef, brier_score_loss, roc_curve, precision_recall_curve,
    confusion_matrix
)

def expected_calibration_error(y_true, y_prob, n_bins=15):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (y_prob >= lo) & (y_prob < hi if i < n_bins - 1 else y_prob <= hi)
        if mask.sum() == 0:
            continue
        acc = (y_true[mask] == (y_prob[mask] >= 0.5)).mean()
        conf = y_prob[mask].mean()
        ece += (mask.sum() / len(y_true)) * abs(acc - conf)
    return float(ece)

def tune_threshold_for_f1(y_val, p_val):
    thresholds = np.linspace(0.01, 0.99, 197)
    best_t, best_f1 = 0.5, -1
    for t in thresholds:
        f1 = f1_score(y_val, (p_val >= t).astype(int))
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t), float(best_f1)

def evaluate(y_true, p_prob, threshold):
    y_pred = (p_prob >= threshold).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, p_prob)),
        "pr_auc": float(average_precision_score(y_true, p_prob)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "brier": float(brier_score_loss(y_true, p_prob)),
        "ece": expected_calibration_error(np.asarray(y_true), np.asarray(p_prob)),
        "threshold": float(threshold),
    }

def plot_diagnostics(y_true, p_prob, threshold, tag, out_dir):
    y_pred = (p_prob >= threshold).astype(int)

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4, 4))
    plt.imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        plt.text(j, i, str(v), ha="center", va="center")
    plt.xticks([0, 1], ["REAL", "FAKE"]); plt.yticks([0, 1], ["REAL", "FAKE"])
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"Confusion Matrix — {tag}")
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, f"confusion_matrix_{tag}.png"), dpi=150); plt.close()

    fpr, tpr, _ = roc_curve(y_true, p_prob)
    plt.figure(figsize=(5, 4)); plt.plot(fpr, tpr); plt.plot([0,1],[0,1],"--",color="gray")
    plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title(f"ROC Curve — {tag}")
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, f"roc_curve_{tag}.png"), dpi=150); plt.close()

    prec, rec, _ = precision_recall_curve(y_true, p_prob)
    plt.figure(figsize=(5, 4)); plt.plot(rec, prec)
    plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title(f"PR Curve — {tag}")
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, f"pr_curve_{tag}.png"), dpi=150); plt.close()

    bins = np.linspace(0, 1, 11)
    bin_ids = np.digitize(p_prob, bins) - 1
    bin_ids = np.clip(bin_ids, 0, 9)
    acc_per_bin, conf_per_bin = [], []
    for b in range(10):
        mask = bin_ids == b
        if mask.sum() == 0:
            acc_per_bin.append(0); conf_per_bin.append((bins[b]+bins[b+1])/2); continue
        acc_per_bin.append((y_true[mask] == (p_prob[mask] >= 0.5)).mean())
        conf_per_bin.append(p_prob[mask].mean())
    plt.figure(figsize=(5, 4))
    plt.plot([0,1],[0,1],"--",color="gray")
    plt.plot(conf_per_bin, acc_per_bin, marker="o")
    plt.xlabel("Mean predicted confidence"); plt.ylabel("Empirical accuracy")
    plt.title(f"Reliability Diagram — {tag}")
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, f"calibration_curve_{tag}.png"), dpi=150); plt.close()

    plt.figure(figsize=(5, 4))
    plt.hist(p_prob, bins=20)
    plt.xlabel("Predicted P(FAKE)"); plt.ylabel("count"); plt.title(f"Confidence Histogram — {tag}")
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, f"confidence_histogram_{tag}.png"), dpi=150); plt.close()


## 9. Model Training — LightGBM (primary) & XGBoost (comparison)
Trained identically across `baseline`, `advanced`, and `mixed` feature sets. Threshold tuned on validation only, then frozen and applied to test.

In [ ]:
import lightgbm as lgb
import xgboost as xgb
import joblib

model_dir = os.path.join(CONFIG["output_dir"], "model")
metrics_dir = os.path.join(CONFIG["output_dir"], "metrics")
fig_dir = os.path.join(CONFIG["output_dir"], "figures")

results = []       # experiment log rows
all_test_probs = {}  # for later error analysis / SHAP on the chosen best model

def train_and_eval(model_name, feature_set):
    Xf, Xv, Xt = feat_fit[feature_set], feat_val[feature_set], feat_test[feature_set]

    if model_name == "lightgbm":
        model = lgb.LGBMClassifier(**CONFIG["lgbm_params"])
        model.fit(Xf, y_fit, eval_set=[(Xv, y_val)],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
    elif model_name == "xgboost":
        model = xgb.XGBClassifier(**CONFIG["xgb_params"], early_stopping_rounds=50)
        model.fit(Xf, y_fit, eval_set=[(Xv, y_val)], verbose=False)
    else:
        raise ValueError(model_name)

    p_val = model.predict_proba(Xv)[:, 1]
    thr, val_f1 = tune_threshold_for_f1(y_val, p_val)

    p_test = model.predict_proba(Xt)[:, 1]
    metrics = evaluate(y_test, p_test, thr)
    metrics.update({"model": model_name, "feature_set": feature_set, "val_f1_at_threshold": val_f1})

    tag = f"{model_name}_{feature_set}"
    plot_diagnostics(y_test, p_test, thr, tag, fig_dir)
    with open(os.path.join(metrics_dir, f"metrics_{tag}.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    joblib.dump(model, os.path.join(model_dir, f"{tag}.joblib"))

    all_test_probs[tag] = p_test
    return metrics, model

trained_models = {}
for feature_set in ["baseline", "advanced", "mixed"]:
    for model_name in ["lightgbm", "xgboost"]:
        print(f"--- Training {model_name} on {feature_set} features ---")
        metrics, model = train_and_eval(model_name, feature_set)
        results.append(metrics)
        trained_models[f"{model_name}_{feature_set}"] = model
        print({k: v for k, v in metrics.items() if k in ["accuracy","f1","roc_auc","pr_auc","mcc"]})

results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(CONFIG["output_dir"], "reports_experiment_a_results.csv")
                   if False else os.path.join(metrics_dir, "all_results.csv"), index=False)
results_df.sort_values("f1", ascending=False)


## 10. Explainability — Feature Importance, SHAP, Permutation Importance
Run on the best-performing configuration (typically `lightgbm` + `mixed`).

In [ ]:
from sklearn.inspection import permutation_importance
import shap

best_row = results_df.sort_values("f1", ascending=False).iloc[0]
best_tag = f"{best_row['model']}_{best_row['feature_set']}"
best_model = trained_models[best_tag]
best_feature_set = best_row["feature_set"]
print("Best config:", best_tag)

shap_dir = os.path.join(CONFIG["output_dir"], "shap")

# --- Native feature importance (fast, tree-based) ---
if best_row["model"] == "lightgbm":
    importances = best_model.booster_.feature_importance(importance_type="gain")
else:
    importances = best_model.feature_importances_
feat_names = [f"f_{i}" for i in range(len(importances))]
imp_df = pd.DataFrame({"feature": feat_names, "importance": importances}).sort_values("importance", ascending=False)
imp_df.to_csv(os.path.join(shap_dir, "feature_importance.csv"), index=False)

plt.figure(figsize=(6, 5))
top = imp_df.head(25)
plt.barh(top["feature"][::-1], top["importance"][::-1])
plt.title(f"Top-25 Feature Importance — {best_tag}")
plt.tight_layout(); plt.savefig(os.path.join(shap_dir, "feature_importance_top25.png"), dpi=150); plt.close()

# --- SHAP (subsampled for speed) ---
X_test_best = feat_test[best_feature_set]
shap_sample_idx = np.random.RandomState(CONFIG["seed"]).choice(len(X_test_best), size=min(1000, len(X_test_best)), replace=False)
X_shap = X_test_best[shap_sample_idx]

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_shap)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

plt.figure()
shap.summary_plot(sv, X_shap, show=False, max_display=20)
plt.tight_layout(); plt.savefig(os.path.join(shap_dir, "shap_summary.png"), dpi=150); plt.close()

# --- Permutation importance (model-agnostic sanity check, on a val subsample) ---
perm_idx = np.random.RandomState(CONFIG["seed"]).choice(len(feat_val[best_feature_set]),
                                                          size=min(2000, len(feat_val[best_feature_set])), replace=False)
perm_result = permutation_importance(best_model, feat_val[best_feature_set][perm_idx], y_val[perm_idx],
                                      n_repeats=5, random_state=CONFIG["seed"], n_jobs=-1, scoring="f1")
perm_df = pd.DataFrame({"feature": feat_names, "importance_mean": perm_result.importances_mean,
                         "importance_std": perm_result.importances_std}).sort_values("importance_mean", ascending=False)
perm_df.to_csv(os.path.join(shap_dir, "permutation_importance.csv"), index=False)

print("Explainability artifacts saved to", shap_dir)


## 11. Error Analysis
False positives, false negatives, highest-confidence mistakes, lowest-confidence predictions — with an image gallery for qualitative review.

In [ ]:
err_dir = os.path.join(CONFIG["output_dir"], "errors")

p_test_best = all_test_probs[best_tag]
thr_best = best_row["threshold"]
y_pred_best = (p_test_best >= thr_best).astype(int)

test_df_reset = test_df.reset_index(drop=True)
err_frame = pd.DataFrame({
    "path": test_df_reset["path"],
    "y_true": y_test,
    "y_pred": y_pred_best,
    "p_fake": p_test_best,
})
err_frame["error_type"] = np.select(
    [ (err_frame.y_true == 0) & (err_frame.y_pred == 1),
      (err_frame.y_true == 1) & (err_frame.y_pred == 0) ],
    ["false_positive", "false_negative"], default="correct"
)
err_frame["confidence"] = np.where(err_frame.y_pred == 1, err_frame.p_fake, 1 - err_frame.p_fake)

err_frame.to_csv(os.path.join(err_dir, "misclassified_samples.csv"), index=False)

fp = err_frame[err_frame.error_type == "false_positive"].sort_values("confidence", ascending=False)
fn = err_frame[err_frame.error_type == "false_negative"].sort_values("confidence", ascending=False)
fp.to_csv(os.path.join(err_dir, "false_positives.csv"), index=False)
fn.to_csv(os.path.join(err_dir, "false_negatives.csv"), index=False)

def gallery(df_subset, title, fname, n=25):
    n = min(n, len(df_subset))
    if n == 0:
        print(f"No samples for {title}"); return
    cols = 5; rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols*2.2, rows*2.4))
    axes = np.array(axes).reshape(-1)
    for i in range(rows*cols):
        ax = axes[i]
        ax.axis("off")
        if i < n:
            row = df_subset.iloc[i]
            img = Image.open(row["path"])
            ax.imshow(img)
            ax.set_title(f"p={row['p_fake']:.2f}", fontsize=8)
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(os.path.join(err_dir, fname), dpi=150)
    plt.close()

gallery(fp.head(25), f"Top False Positives — {best_tag}", "false_positive_gallery.png")
gallery(fn.head(25), f"Top False Negatives — {best_tag}", "false_negative_gallery.png")

low_conf = err_frame.sort_values("confidence").head(25)
gallery(low_conf, f"Lowest-Confidence Predictions — {best_tag}", "low_confidence_gallery.png")

print(f"FP: {len(fp)}, FN: {len(fn)} out of {len(err_frame)} test samples")


## 12. Experiment Log & Summary Report

In [ ]:
log_path = os.path.join(CONFIG["output_dir"], "report", "experiment_a_report.md")

summary_table = results_df[["model","feature_set","accuracy","precision","recall","f1",
                             "roc_auc","pr_auc","balanced_accuracy","mcc","brier","ece","threshold"]]\
                .sort_values("f1", ascending=False)

with open(log_path, "w") as f:
    f.write(f"# Experiment A Report — {CONFIG['experiment_id']}\n\n")
    f.write(f"Seed: {CONFIG['seed']}  \n")
    f.write(f"Best config: **{best_tag}**  \n\n")
    f.write("## Results\n\n")
    f.write(summary_table.to_markdown(index=False))
    f.write("\n\n## Discussion\n")
    f.write("- Which features dominate: see `shap/feature_importance.csv` and `shap/shap_summary.png`.\n")
    f.write("- Failure modes: see `errors/false_positive_gallery.png` and `errors/false_negative_gallery.png`.\n")
    f.write("- Calibration: see `figures/calibration_curve_*.png` — check whether predicted confidence tracks empirical accuracy.\n")

print(summary_table.to_string(index=False))
print("\nFull report written to", log_path)

# append to a master cross-experiment log if it exists (create if not)
master_log_path = "/kaggle/working/experiment_logs.csv"
log_rows = summary_table.copy()
log_rows.insert(0, "experiment_id", CONFIG["experiment_id"])
if os.path.exists(master_log_path):
    existing = pd.read_csv(master_log_path)
    combined = pd.concat([existing, log_rows], ignore_index=True)
else:
    combined = log_rows
combined.to_csv(master_log_path, index=False)
print("Master experiment log updated:", master_log_path)


## 13. Outputs Summary

All artifacts are saved under `/kaggle/working/experiment_A/`:

```
experiment_A/
├── model/            *.joblib  (one per model x feature-set combo)
├── metrics/          metrics_*.json, all_results.csv
├── figures/           confusion_matrix / roc_curve / pr_curve / calibration_curve / confidence_histogram (per combo)
│                      sample_images.png, feat_variance_*.png, pca_*.png
├── shap/             feature_importance.csv/.png, shap_summary.png, permutation_importance.csv
├── errors/           misclassified_samples.csv, false_positives.csv, false_negatives.csv, *_gallery.png
├── logs/             experiment_config.json, split_summary.json, feature_analysis.json
├── features/         fit.npz, val.npz, test.npz  (cached feature matrices — reusable for Experiment D fusion)
└── report/           experiment_a_report.md
```

**Next steps:** Experiment B (ConvNeXt) and Experiment C (ViT) should reuse this exact `fit_df` / `val_df` / `test_df` split and the same evaluation utilities (Section 8) — copy them verbatim rather than re-deriving, so any performance delta is attributable to the model, not the pipeline.
